### Cuaderno de simulaciones con Schelling 3 agentes

Cargamos los paquetes necesarios (en mi portatel se debe ejecutar en el entorno mesa-env en WSL)

In [1]:
!pip install mesa solara

In [2]:
import mesa
import random
from mesa.visualization import SolaraViz, make_space_component
from ipywidgets import interact, IntSlider, FloatSlider
import solara
from IPython.display import display, HTML

/usr/local/lib/python3.12/dist-packages/solara/validate_hooks.py:122: UserWarning: /usr/local/lib/python3.12/dist-packages/mesa/visualization/solara_viz.py:399: ComponentsView: `use_state` found despite early return on line 376
To suppress this check, replace the line with:
    current_tab_index, set_current_tab_index = solara.use_state(0)  # noqa: SH101

Make sure you understand the consequences of this, by reading about the rules of hooks at:
    https://solara.dev/documentation/advanced/understanding/rules-of-hooks

  warnings.warn(str(e))


# Cargamos las funcines del modelo se Schelling con 3 poblaciones

En esta celda cargamos el agente **Persona** y sus funciones (**incomodidad**, **move**)

In [3]:
# ========== AGENTE ==========
class Persona(mesa.Agent):
    def __init__(self, model, tipo: int, se_ha_movido: bool) -> None:
        super().__init__(model)
        self.tipo = tipo  # tipos (-1), (+1), hostiles y 0 neutrales
        self.se_ha_movido = se_ha_movido # para actualizar TRUE cuando el agente cambia de posición


# La incomodidad es de un agente (self) en una casilla (pos) asumiendo que el agente es de un tipo. incomodidad(agente,posición,tipo) es el número
# de vecinos incómodos que tendría el agente si se encontrara en la posición.

    def incomodidad(self, pos, tipo):
        contador = 0
        vecindad = self.model.grid.get_neighbors(pos, moore=True, include_center=False)
        if tipo == 1:
            for vecino in vecindad:
                if vecino.tipo == -1: contador += 1
        elif tipo == -1:
            for vecino in vecindad:
                if vecino.tipo == 1: contador += 1
        elif tipo == 0:
            for vecino in vecindad:
                contador += abs(vecino.tipo)
        return contador

# Esta es la función fundamental que recoloca a un agente, siempre que su incomodidad esté por encima de su tolerancia.
# Para ello busca la casilla vacía más cercana en la que tendría una incomodidad menor a la actual, y se mueve ahí.
# Si no encuentra ninguna casilla con mejor situación, se resigna y se queda donde está.

    def move(self):
        self.se_ha_movido = False
        possible_steps = []
        radio = 0
        max_iteraciones = self.model.grid.width // 2 + 1
        incomodidad_base = self.incomodidad(self.pos, self.tipo)

        if incomodidad_base > self.model.tolerance:
            while not possible_steps and radio < max_iteraciones:
                radio += 1
                vecindario = self.model.grid.get_neighborhood(
                    self.pos, moore=True, include_center=False, radius=radio
                )
                empty_steps = [step for step in vecindario if self.model.grid.is_cell_empty(step)]
                possible_steps = [step for step in empty_steps if self.incomodidad(step, self.tipo) < incomodidad_base]

        if possible_steps:
            new_position = random.choice(possible_steps)
            self.model.grid.move_agent(self, new_position)
            self.se_ha_movido = True




En esta celda definimos el modelo

  Modelo_Desplazamiento(N1,N2,N3, width, height, tol)

  N1 := población tipo (+1)
  N2 := población tipo (0)
  N3 := población tipo (-1)
  tol := los agentes se desplazan cuando incomodidad > tol

  **contact_measure**

  
  

In [4]:
# ========== MODELO ==========

class Modelo_Desplazamiento(mesa.Model):
    def __init__(self, N1, N2, N3, width, height, tol, seed=None):
        super().__init__(seed=seed)
        self.num_agents = N1 + N2 + N3
        self.grid = mesa.space.SingleGrid(width, height, True)
        self.running = True
        self.tolerance = tol
        self.steps_to_equilibrium = 0
        self.in_equilibrium = False
        self.contact_measure = {}
        self.datacollector = mesa.DataCollector(
            model_reporters={
                "Steps_to_Equilibrium": "steps_to_equilibrium",
                "Contact_Measure": "contact_measure",
                "In_Equilibrium": "in_equilibrium"
            }
        )

        for i in range(N1): Persona(model=self, tipo=1, se_ha_movido=False)
        for i in range(N2): Persona(model=self, tipo=0, se_ha_movido=False)
        for i in range(N3): Persona(model=self, tipo=-1, se_ha_movido=False)


        # Obtener todas las posiciones del grid
        all_positions = [
            (x, y)
            for x in range(self.grid.width)
            for y in range(self.grid.height)
        ]

        # Mezclar aleatoriamente
        self.random.shuffle(all_positions)

        # Asignar una posición por agente
        for agent, pos in zip(self.agents, all_positions):
            self.grid.place_agent(agent, pos)

        #placed_agents = set()
        #for i in range(self.num_agents):
        #    agent = self.agents[i]
        #    x, y = random.randrange(self.grid.width), random.randrange(self.grid.height)
        #    while (x, y) in placed_agents:
        #        x, y = random.randrange(self.grid.width), random.randrange(self.grid.height)
        #    placed_agents.add((x, y))
        #    self.grid.place_agent(agent, (x, y))

        self.calculate_contact_measure()

    def calculate_contact_measure(self):
        """Calcula la medida de contacto desagregada por tipo de agente"""
        contact_counts = {
            -1: {-1: 0, 0: 0, 1: 0},
             0: {-1: 0, 0: 0, 1: 0},
             1: {-1: 0, 0: 0, 1: 0},
        }
        Total_contacts = 0

        for agent in self.agents:
            neighbors = self.grid.get_neighbors(agent.pos, moore=True, include_center=False)
            for neighbor in neighbors:
                contact_counts[agent.tipo][neighbor.tipo] += 1
                Total_contacts += 1

        Total_contacts = Total_contacts / 2

        for i in [-1,0,1]: contact_counts[i][i] = contact_counts[i][i] / 2

        # Guardamos proporciones
        self.contact_measure = {}
        for agent_type in [-1, 0, 1]:
            self.contact_measure[agent_type] = {}
            for neighbor_type in [-1, 0, 1]:
                if Total_contacts > 0:
                    self.contact_measure[agent_type][neighbor_type] = (
                        contact_counts[agent_type][neighbor_type] / Total_contacts
                    )
                else:
                    self.contact_measure[agent_type][neighbor_type] = 0

    def check_equilibrium(self) -> bool:
        """Verifica si el modelo ha alcanzado el equilibrio"""
        for agent in self.agents:
            if agent.se_ha_movido:
                return False
        return True

    def step(self):
        if not self.in_equilibrium:
            self.agents.shuffle_do("move")
            self.calculate_contact_measure()

            if self.check_equilibrium():
                self.in_equilibrium = True
                self.running = False
            else:
                self.steps_to_equilibrium += 1

            self.datacollector.collect(self)


Esta celda es para visualización del modelo, con sliders

In [5]:
# ========== VISUALIZACIÓN ==========
def agent_portrayal(agent):
    portrayal = {"size": 50}
    if agent.tipo == 1:
        portrayal["color"] = "tab:green"
    elif agent.tipo == 0:
        portrayal["color"] = "tab:orange"
    elif agent.tipo == -1:
        portrayal["color"] = "tab:red"
    else:
        portrayal["color"] = "gray"
    return portrayal

def InfoComponent(model):
    """Componente que muestra información del modelo"""
    if model is None:
        return solara.Markdown("## Esperando inicio del modelo...")

    if model.in_equilibrium:
        status_text = "**EQUILIBRIO ALCANZADO** ✅"
        status_color = "green"
    else:
        status_text = "En proceso..."
        status_color = "orange"

    if isinstance(model.contact_measure, dict):
        header = "Tipo agente |  -1   |   0   |   1  "
        sep = "-" * len(header)
        rows = "\n".join(
            f"     {a:>2}     | " + " | ".join(f"{model.contact_measure[a][b]:.2f}" for b in [-1, 0, 1])
            for a in [-1, 0, 1]
        )
        contact_info = f"```\n{header}\n{sep}\n{rows}\n```"
    else:
        contact_info = f"{model.contact_measure:.4f}"
    return solara.Card(
        title="Información del Modelo",
        children=[
            solara.Markdown(f"""
            **Estado del Modelo**

            - **Pasos hasta el equilibrio:** {model.steps_to_equilibrium}
            - **Medida de contacto (proporciones):** {contact_info}
            - **Estado:** <span style='color: {status_color}'>{status_text}</span>
            - **Tolerancia:** {model.tolerance}
            - **Agentes:** {model.num_agents}
              (Tipo 1: {sum(1 for a in model.agents if a.tipo == 1)},
              Tipo 0: {sum(1 for a in model.agents if a.tipo == 0)},
              Tipo -1: {sum(1 for a in model.agents if a.tipo == -1)})
            """)
        ]
    )



Celda de visualización del modelo

In [10]:
def ejecutar_modelo(N1=500, N2=500, N3=500, tol=2.0):
    width = 40
    height = 40
    max_celdas = width * height

    if N1 + N2 + N3 > max_celdas:
        return display(HTML(f"<b style='color:red;'>Error:</b> "
                            f"N1 + N2 + N3 = {N1+N2+N3} supera las {max_celdas} celdas de la grilla."))

    model_params = {"N1": N1, "N2": N2, "N3": N3, "width": width, "height": height, "tol": tol}
    model = Modelo_Desplazamiento(**model_params)

    SpaceGraph = make_space_component(agent_portrayal)

    page = SolaraViz(
        model,
        components=[SpaceGraph, InfoComponent],
        model_params=model_params,
        name="Schelling desplazamiento",
    )

    return page

# INTERACTIVO
interact(
    ejecutar_modelo,
    N1=IntSlider(min=0, max=533, step=1, value=500, description="N1"),
    N2=IntSlider(min=0, max=533, step=1, value=500, description="N2"),
    N3=IntSlider(min=0, max=533, step=1, value=500, description="N3"),
    tol=FloatSlider(min=0, max=10, step=0.5, value=2, description="Tol"),
)


interactive(children=(IntSlider(value=500, description='N1', max=533), IntSlider(value=500, description='N2', …

<function __main__.ejecutar_modelo(N1=500, N2=500, N3=500, tol=2.0)>

In [8]:
ejecutar_modelo(
    N1=500,
    N2=500,
    N3=500,
    tol=2
    )

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text